<a href="https://colab.research.google.com/github/MahuL88/Latihan/blob/main/Latihan_Hyperparameter_Tuning_Dicoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latihan Regresi   

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.4 MB/s eta 0:00:00


## Load Data

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
import time
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from skopt import BayesSearchCV
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


# Mengunduh dataset California Housing
X, y = fetch_california_housing(return_X_y=True)

# Membagi dataset menjadi training set dan testing set (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Melakukan scaling pada data (karna menggunakana regresi)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Shape of training data:", X_train.shape)
print("Shape of testing data:", X_test.shape)

Shape of training data: (14448, 8)
Shape of testing data: (6192, 8)


## Latih Data

In [ ]:
# Inisialisasi model Random Forest Regressor
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

# Evaluasi awal model tanpa tuning
y_pred = rf.predict(X_test)
initial_mse = mean_squared_error(y_test, y_pred)
print(f"Initial MSE on test set (without tuning): {initial_mse:.2f}")

Initial MSE on test set (without tuning): 0.26


## Hyperparameter Tuning


### GridSearch

In [ ]:
start_time = time.time()  # Mencatat waktu mulai

# Definisikan parameter grid untuk Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False] #sampling with replacement
}

# Inisialisasi GridSearchCV dengan menggunakan 1 core CPU dan logging dengan menampilkan semua informasi
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=1, verbose=2)
grid_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Grid Search): {grid_search.best_params_}")
best_rf_grid = grid_search.best_estimator_

# Evaluasi performa model setelah Grid Search
y_pred_grid = best_rf_grid.predict(X_test)
grid_search_mse = mean_squared_error(y_test, y_pred_grid)
print(f"MSE after Grid Search: {grid_search_mse:.2f}")
end_time = time.time()  # Mencatat waktu selesai
execution_time = end_time - start_time  # Menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Terlihat bahwa nilai MSE setelah melakukan Grid search setelah menemukan hyperparameter terbaik adalah 0.25 dan waktu untuk eksekusi seluruh hyperparameter ialah 11192 detik atau setara dengan 3 jam 6 menit 32 detik.

### RandomSearch

In [ ]:
start_time = time.time()  # Mencatat waktu mulai

# Definisikan ruang pencarian untuk Random Search
param_dist = {
    'n_estimators': np.arange(100, 500, 100),
    'max_depth': [None] + list(np.arange(10, 50, 10)),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 5),
    'bootstrap': [True, False]
}

# Inisialisasi RandomizedSearchCV dengan menggunakan seluruh core CPU, hanya menggunakan 5 kombinasi acak, dan logging dengan menampilkan semua informasi
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=5, cv=3, n_jobs=-1, verbose=2, random_state=42)
random_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Random Search): {random_search.best_params_}")
best_rf_random = random_search.best_estimator_

# Evaluasi performa model setelah Random Search
y_pred_random = best_rf_random.predict(X_test)
random_search_mse = mean_squared_error(y_test, y_pred_random)
print(f"MSE after Grid Search: {random_search_mse:.2f}")
end_time = time.time()  # Mencatat waktu selesai
execution_time = end_time - start_time  # Menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Terlihat bahwa nilai MSE setelah melakukan Random Search setelah menemukan hyperparameter terbaik adalah 0.25 yang dimana nilai ini sama dengan grid search(tapi tidak semua kasus akan sama) dan waktu untuk eksekusi seluruh hyperparameter ialah 449 detik atau setara dengan 7 menit 29 detik.

### Bayesian Optimization

In [ ]:
start_time = time.time()  # Mencatat waktu mulai

# Definisikan ruang pencarian untuk Bayesian Optimization
param_space = {
    'n_estimators': (100, 500),
    'max_depth': (10, 50),
    'min_samples_split': (2, 10),
    'min_samples_leaf': (1, 4),
    'bootstrap': [True, False]
}

# Inisialisasi BayesSearchCV dengan menggunakan 1 core CPU, dengan 32 kali iterasi, dan logging dengan menampilkan semua informasi
bayes_search = BayesSearchCV(estimator=rf, search_spaces=param_space, n_iter=32, cv=3, n_jobs=1, verbose=2, random_state=42)
bayes_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Bayesian Optimization): {bayes_search.best_params_}")
best_rf_bayes = bayes_search.best_estimator_

# Evaluasi performa model setelah Random Search
y_pred_bayes = best_rf_bayes.predict(X_test)
bayes_mse = mean_squared_error(y_test, y_pred_bayes)
print(f"MSE after Grid Search: {bayes_mse:.2f}")
end_time = time.time()  # Mencatat waktu selesai
execution_time = end_time - start_time  # Menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END bootstrap=True, max_depth=39, min_samples_leaf=4, min_samples_split=5, n_estimators=368; total time=  39.3s
[CV] END bootstrap=True, max_depth=39, min_samples_leaf=4, min_samples_split=5, n_estimators=368; total time=  53.0s
[CV] END bootstrap=True, max_depth=39, min_samples_leaf=4, min_samples_split=5, n_estimators=368; total time=  45.8s
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END bootstrap=False, max_depth=45, min_samples_leaf=2, min_samples_split=10, n_estimators=446; total time= 1.5min
[CV] END bootstrap=False, max_depth=45, min_samples_leaf=2, min_samples_split=10, n_estimators=446; total time= 1.2min
[CV] END bootstrap=False, max_depth=45, min_samples_leaf=2, min_samples_split=10, n_estimators=446; total time= 1.4min
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END bootstrap=True, max_depth=47, min_samples_leaf=1, min_samples_split=5, n_estimators=175; total time=  23.4

Terlihat bahwa nilai MSE setelah melakukan Bayesian Optimization setelah menemukan hyperparameter terbaik adalah 0.25 yang dimana nilai ini sama dengan grid search dan random search(tapi tidak semua kasus akan sama) dan waktu untuk eksekusi seluruh hyperparameter ialah 4562 detik atau setara dengan 1 jam 16 menit 2 detik. berbeda dengan metode sebelumnya, bayes ini hanya melakukan 1 kombinasi yg berulang kali dihitung setiap iterasi dan memanfaatkan informasi dari iterasi sebelumnya untuk mencari kombinasi hyperparameter terbaik.

# **Kesimpulan**



*   Grid Search memberikan hasil optimal tetapi membutuhkan waktu komputasi yang lebih lama
*   Random Search bisa saja kehilangan kombinasi hyperparameter yang optimal tapi cepat jika digunakan pada kasus lain
*   Bayesian Optimization memberikan keseimbangan terbaik antara efisiensi komputasi dan hasil optimal(meskipun pada kasus ini sama)


# Latihan Klasifikasi

## Load Data

In [ ]:
# Mengunduh dataset German Credit dari OpenML
X, y = fetch_openml(name='credit-g', version=1, return_X_y=True, as_frame=True)

# Konversi target menjadi numerik
le = LabelEncoder()
y = le.fit_transform(y)  # Mengubah 'good' menjadi 1 dan 'bad' menjadi 0

# Melakukan One-Hot Encoding pada fitur kategorikal
X_encoded = pd.get_dummies(X, drop_first=True)  # Konversi fitur kategorikal menjadi numerik

# Membagi dataset menjadi training set dan testing set (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42)

print("Shape of training data:", X_train.shape)
print("Shape of testing data:", X_test.shape)

Shape of training data: (700, 48)
Shape of testing data: (300, 48)


## Latih Data

In [ ]:
# Inisialisasi model Random Forest tanpa hyperparameter tuning
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Evaluasi awal model tanpa tuning
initial_score = rf.score(X_test, y_test)
print(f"Initial accuracy on test set (without tuning): {initial_score:.2f}")

Initial accuracy on test set (without tuning): 0.76


## Hyperparameter Tuning

### GridSearch

In [ ]:
# Definisikan parameter grid untuk Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

# Inisialisasi GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Grid Search): {grid_search.best_params_}")
best_rf_grid = grid_search.best_estimator_

# Evaluasi performa model pada test set
grid_search_score = best_rf_grid.score(X_test, y_test)
print(f"Accuracy after Grid Search: {grid_search_score:.2f}")

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best parameters (Grid Search): {'criterion': 'gini', 'max_depth': 30, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy after Grid Search: 0.76


Terlihat bahwa nilai akurasi setelah melakukan Grid search setelah menemukan hyperparameter terbaik adalah 0.76 yang memiliki nilai yang sama dengan model tanpa tuning.

### RandomSearch

In [ ]:
# Definisikan ruang pencarian untuk Random Search
param_dist = {
    'n_estimators': np.linspace(100, 500, 5, dtype=int),
    'max_depth': np.linspace(10, 50, 5, dtype=int),
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

# Inisialisasi RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=20, cv=3, n_jobs=-1, verbose=2, random_state=42)
random_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Random Search): {random_search.best_params_}")
best_rf_random = random_search.best_estimator_

# Evaluasi performa model pada test set
random_search_score = best_rf_random.score(X_test, y_test)
print(f"Accuracy after Random Search: {random_search_score:.2f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters (Random Search): {'n_estimators': np.int64(200), 'min_samples_split': 5, 'max_depth': np.int64(30), 'criterion': 'gini'}
Accuracy after Random Search: 0.76


Terlihat bahwa nilai akurasi setelah melakukan Random search setelah menemukan hyperparameter terbaik juga 0.76 yang memiliki nilai yang sama dengan tanpa tuning dan gridsearch.

### Bayesian Optimization

In [ ]:
# Definisikan ruang pencarian untuk Bayesian Optimization
param_space = {
    'n_estimators': (100, 500),
    'max_depth': (10, 50),
    'min_samples_split': (2, 10),
    'criterion': ['gini', 'entropy']
}

# Inisialisasi BayesSearchCV
bayes_search = BayesSearchCV(estimator=rf, search_spaces=param_space, n_iter=32, cv=3, n_jobs=-1, verbose=2, random_state=42)
bayes_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Bayesian Optimization): {bayes_search.best_params_}")
best_rf_bayes = bayes_search.best_estimator_

# Evaluasi performa model pada test set
bayes_search_score = best_rf_bayes.score(X_test, y_test)
print(f"Accuracy after Bayesian Optimization: {bayes_search_score:.2f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

Terlihat bahwa nilai akurasi setelah melakukan bayesiam optimization setelah menemukan hyperparameter terbaik adalah 0.75 yang turun 0.01 dari model sebelum-sebelumnya. terlihat bahwa performa model tidak berubah sama sekali atau bahkan menurun.

# **Kesimpulan**

penurunan akurasi setelah hyperparameter tuning terjadi karena ; overfit,underfit, pemilihan Ruang Pencarian yg terlalu kecil atau besar(tidak mencakup area yang optimal), dan cross-validation yang kurang efektif.

untuk kasus ini, salah satu alasan model yang dibangun tidak berubah adalah akrena dataset yang digunakan terlalu kecil atau dummy. sehingga, perlu dipastikan bahwa datasetnya yg digunakan baik, perluas dan fokuskan ruang pencarian hyperparameter pada yang berdampak, dan menggunakan lebih banyak fold pada cv.